# Aula 06 · Otimização

Esta aula apresenta o [capítulo 6 do site](https://lacouth.github.io/metodos_telecom-site/unidade3-raizes/06-otimizacao/). A ideia central: **no topo do morro e no fundo do vale, a derivada é zero** — otimizar é achar uma raiz da derivada, e as ferramentas dos capítulos 2 e 5 já bastam.

**Ao fim da aula você consegue:**

1. usar uma varredura numa grade para ver a forma da função e achar um bom chute;
2. deduzir Newton para otimização e aplicá-lo com as derivadas numéricas;
3. distinguir mínimo de máximo pelo sinal de $f''$, e local de global pela varredura;
4. transformar um problema de máximo num de mínimo.

**Roteiro:** 🧩 · 1. varredura · 2. 🧑‍🏫 Newton para otimização · 3. quando o ótimo engana · 4. confira · 5. outra área · 🎯 prática · 🧩 o arremesso · 📋 a lista · 🚪

O caderno ocupa mais de um encontro: pare onde a aula terminar e continue daí na
seguinte.

## Como usar este caderno

- **Rode a célula ⚙️** logo abaixo antes de tudo (e de novo se o Colab reiniciar).
- **🧑‍🏫 No quadro:** a dedução é feita à mão, no quadro. Acompanhe **no seu
  caderno de papel** — é o mesmo tipo de conta que cai na parte em papel da prova.
  O resumo fica recolhido aqui, para conferir depois.
- **🧰 Comando novo:** antes do primeiro uso de um comando de `numpy` ou
  `matplotlib`, uma caixa explica o que ele faz. Rode a célula de exemplo logo
  abaixo dela.
- **✍️ Passo:** o código é escrito ao vivo, em pedaços pequenos — a instrução
  está logo acima de cada célula vazia. Estudando sozinho, escreva você mesmo; o
  código completo está no capítulo do site (links 📖).
- Depois de escrever e **antes de rodar**, registre a sua previsão. Só então rode
  e abra o **▶ O que aconteceu**. A previsão errada é a parte que ensina — não a
  apague.
- **🎯 Sua vez:** escreva a função no lugar de `# sua solução aqui` e rode a
  célula `confere` logo abaixo: ✅ acertou, ❌ ainda não. Tente antes de abrir a
  💡 Dica.
- O caderno pode ocupar mais de uma aula: continue de onde parou, rodando antes a
  célula ⚙️ e as células 📦.

In [ ]:
# ⚙️ Rode esta célula antes de tudo. Ela prepara a correção automática dos
# exercícios 🎯 — não precisa ler (usa coisas que não fazem parte do curso).
import math


def _mostra(argumentos):
    textos = []
    for a in argumentos:
        textos.append(a.__name__ if callable(a) else repr(a))
    return ", ".join(textos)


def _numero(x):
    try:
        float(x)
        return not isinstance(x, (str, bool))
    except (TypeError, ValueError):
        return False


def _igual(veio, esperado, tol):
    # Número: compara com tolerância relativa, porque conta com float quase
    # nunca bate na última casa. Lista, tupla ou array: item a item.
    if _numero(esperado) and _numero(veio):
        return math.isclose(float(veio), float(esperado), rel_tol=tol, abs_tol=tol)
    if isinstance(esperado, (list, tuple)) and hasattr(veio, "__len__") and not isinstance(veio, str):
        if len(veio) != len(esperado):
            return False
        return all(_igual(v, e, tol) for v, e in zip(veio, esperado))
    return veio == esperado


def confere(funcao, casos, tol=1e-6):
    """Chama funcao com cada caso (argumentos, esperado) e diz se acertou."""
    certos = 0
    for numero, (argumentos, esperado) in enumerate(casos, start=1):
        chamada = f"{funcao.__name__}({_mostra(argumentos)})"
        try:
            veio = funcao(*argumentos)
        except Exception as erro:
            print(f"❌ {chamada} deu erro: {type(erro).__name__}: {erro}")
            continue
        if veio is not None and _igual(veio, esperado, tol):
            certos += 1
            print(f"✅ {chamada} devolveu {veio!r}")
        elif veio is None:
            print(f"❌ {chamada} devolveu None — faltou o return?")
        else:
            print(f"❌ {chamada} devolveu {veio!r}, mas devia ser {esperado!r}")
    print(f"{certos} de {len(casos)} certos")


def confere_valor(nome, valor, esperado, tol=1e-6):
    """Diz se a variável `nome` ficou com o valor esperado."""
    if valor is not None and _igual(valor, esperado, tol):
        print(f"✅ {nome} = {valor!r}")
    else:
        print(f"❌ {nome} vale {valor!r}, mas devia ser {esperado!r}")

# --- bibliotecas desta aula ---
import numpy as np
import matplotlib.pyplot as plt

## 🧩 O problema da aula

> **Atletismo — o ângulo do arremesso de peso.**
>
> *Todo livro de física diz que o alcance máximo é com 45°. O treinador de uma
> atleta de arremesso de peso desconfia: o peso sai da mão dela a 2,1 m do chão, e
> não do nível do chão. "**Qual o ângulo ideal para ela, e quantos centímetros ela
> ganha em relação aos 45°?**" Num campeonato, poucos centímetros separam o ouro
> da prata.*

O alcance como função do ângulo tem fórmula, mas o máximo dela não sai fácil no
papel. No fim da aula, você o acha.

## 1. Primeiro, uma varredura

Uma lata de 350 mL pode ser alta e fina ou baixa e larga. Com o volume fixo, a área
de chapa de alumínio depende só do raio: $A(r) = 2\pi r^2 + 700/r$. O jeito mais
simples de achar o mínimo é calcular $A$ numa grade de raios e pegar o menor.

📖 [capítulo 6 · Primeiro, uma varredura](https://lacouth.github.io/metodos_telecom-site/unidade3-raizes/06-otimizacao/#primeiro-uma-varredura)

In [ ]:
# 📦 dados prontos — só rode esta célula
# Área de chapa (cm²) de uma lata de 350 mL com raio r (cm).
V = 350.0


def area(r):
    return 2 * np.pi * r**2 + 2 * V / r

> 🧰 **Comando novo: `np.argmin`**
>
> `np.argmin(a)` devolve a **posição** do menor elemento do array — o irmão do
> `np.argmax` da Aula 00.

In [ ]:
# 🧰 exemplo — só rode e veja a saída
custos = np.array([8.2, 5.1, 3.9, 4.4, 6.0])
print(np.min(custos), np.argmin(custos))

**✍️ Passo 1.** Crie `raios = np.linspace(1, 10, 91)`, calcule `areas = area(raios)`, ache a posição `k` do menor com `np.argmin` e imprima `raios[k]` e `areas[k]`.

In [ ]:
# ✍️ passo 1

**Preveja:** o raio que gasta menos chapa é maior ou menor que o de uma lata comum (3,3 cm)?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

`3.8` cm, com 274,94 cm² de chapa: **maior** que o da lata comum. A precisão é a
da grade (0,1 cm) — boa para um chute, pouca para uma resposta.

📖 [capítulo 6 · Primeiro, uma varredura](https://lacouth.github.io/metodos_telecom-site/unidade3-raizes/06-otimizacao/#primeiro-uma-varredura)

</details>

## 2. No quadro: o mínimo é uma raiz da derivada

📖 [capítulo 6 · No quadro: o mínimo é uma raiz da derivada](https://lacouth.github.io/metodos_telecom-site/unidade3-raizes/06-otimizacao/#no-quadro-o-minimo-e-uma-raiz-da-derivada)

### 🧑‍🏫 No quadro — Newton para otimização

Caderno de papel aberto. No quadro:

1. no mínimo ou no máximo, $f'(x^*) = 0$;
2. Newton na função $g = f'$, com $g' = f''$;
3. as duas derivadas pelas diferenças centrais do capítulo 2;
4. o sinal de $f''$: vale ou morro.

<details>
<summary><b>▶ O resumo do quadro</b></summary>

$$ x_{i+1} = x_i - \frac{f'(x_i)}{f''(x_i)}, \qquad
f' \approx \frac{f(x+h) - f(x-h)}{2h}, \qquad
f'' \approx \frac{f(x+h) - 2f(x) + f(x-h)}{h^2} $$

$f'' > 0$: mínimo. $f'' < 0$: máximo.

</details>

**✍️ Passo 2.** Com `r = 3.8` e `h = 1e-4`, calcule `d1` e `d2` (as diferenças centrais de `area`), faça `r = r - d1 / d2` e imprima `r`. Repita mais duas vezes.

In [ ]:
# ✍️ passo 2

**Preveja:** quantas casas o raio ganha a cada passo?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

$3{,}8190$, $3{,}819115$, $3{,}8191149$: em três passos, 10 casas. A
resposta exata é $(V/2\pi)^{1/3} = 3{,}8191$ cm, com a altura **igual ao
diâmetro**. Latas reais são mais altas: o fundo usa chapa mais grossa e a lata
precisa caber na mão. O modelo responde à pergunta feita.

📖 [capítulo 6 · No quadro: o mínimo é uma raiz da derivada](https://lacouth.github.io/metodos_telecom-site/unidade3-raizes/06-otimizacao/#no-quadro-o-minimo-e-uma-raiz-da-derivada)

</details>

> ⚠️ **Armadilha.** Com `h = 1e-8` na segunda derivada, o $h^2$ do denominador vira $10^{-16}$ e o
arredondamento toma conta (capítulo 2). Para $f''$, use $h$ perto de $10^{-4}$.

### 🎯 Sua vez — Um passo de Newton para otimização

Escreva `passo_otimizacao(f, x)`, que devolve o próximo ponto, com $f'$ e $f''$ pelas diferenças centrais e `h = 1e-4`.

In [ ]:
def passo_otimizacao(f, x):
    # sua solução aqui
    pass

In [ ]:
confere(passo_otimizacao, [
    ((area, 3.8), 3.8190189214087447),
])

<details>
<summary><b>💡 Dica</b></summary>

São as três linhas do passo 2 dentro de uma função, com `return`.

</details>

## 3. Quando o ótimo não é o que se queria

$f(x) = x^4 - 3x^2 + x$ tem dois vales e um morro entre eles. Newton procura um
ponto com $f' = 0$ — qualquer um.

📖 [capítulo 6 · Quando o ótimo não é o que se queria](https://lacouth.github.io/metodos_telecom-site/unidade3-raizes/06-otimizacao/#quando-o-otimo-nao-e-o-que-se-queria)

In [ ]:
# 📦 dados prontos — só rode esta célula
def f4(x):
    return x**4 - 3 * x**2 + x

**✍️ Passo 3.** Aplique 30 passos de Newton para otimização (as contas do passo 2) em `f4` a partir de `x = 0.1`. Imprima `x`, `f4(x)` e a segunda derivada no ponto final.

In [ ]:
# ✍️ passo 3

**Preveja:** Newton vai achar um mínimo?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Não: acha $x = 0{,}17$, com $f'' = -5{,}65 < 0$ — um **máximo**. Partindo de
2, acharia o vale da direita ($f = -1{,}07$), que não é o mais fundo (o global
está em $-1{,}30$, com $f = -3{,}51$). Newton não distingue vale de morro,
nem vale raso de vale fundo.

📖 [capítulo 6 · Quando o ótimo não é o que se queria](https://lacouth.github.io/metodos_telecom-site/unidade3-raizes/06-otimizacao/#quando-o-otimo-nao-e-o-que-se-queria)

</details>

### 🎯 Sua vez — Vale ou morro?

Escreva `tipo_de_ponto(f, x)`, que devolve `"minimo"` se a segunda derivada central (`h = 1e-4`) for positiva em `x`, e `"maximo"` caso contrário.

In [ ]:
def tipo_de_ponto(f, x):
    # sua solução aqui
    pass

In [ ]:
confere(tipo_de_ponto, [
    ((f4, -1.3008), "minimo"),
    ((f4, 0.1699), "maximo"),
    ((f4, 1.1309), "minimo"),
])

<details>
<summary><b>💡 Dica</b></summary>

A segunda derivada central e um `if`.

</details>

## 4. Confira com a biblioteca

📖 [capítulo 6 · Confira com a biblioteca](https://lacouth.github.io/metodos_telecom-site/unidade3-raizes/06-otimizacao/#confira-com-a-biblioteca)

> 🧰 **Comando novo: `from scipy.optimize import minimize_scalar`**
>
> `minimize_scalar(f, bounds=(a, b), method="bounded")` acha o mínimo de `f` no
> intervalo `[a, b]` (`bounds=`), sem sair dele (`method="bounded"`). O resultado é
> um "pacote": o ponto ótimo fica em `.x` e o valor mínimo, em `.fun`. Para um
> **máximo**, minimize $-f$.

In [ ]:
# 🧰 exemplo — só rode e veja a saída
from scipy.optimize import minimize_scalar

resultado = minimize_scalar(f4, bounds=(-2, 0), method="bounded")
print(resultado.x, resultado.fun)

**✍️ Passo 4.** Importe o `minimize_scalar` e confira o mínimo da lata com `bounds=(1, 10)`. Imprima `.x` e `.fun`.

In [ ]:
# ✍️ passo 4

**Preveja:** bate com o Newton do passo 2?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Bate: $r = 3{,}8191$ cm e $A = 274{,}93$ cm².

📖 [capítulo 6 · Confira com a biblioteca](https://lacouth.github.io/metodos_telecom-site/unidade3-raizes/06-otimizacao/#confira-com-a-biblioteca)

</details>

## 5. Mesmo método, outra área

**Eletrônica.** Uma fonte de 12 V com resistência interna de 50 Ω alimenta uma
carga $R$. A potência na carga é $P(R) = V^2R/(50 + R)^2$. Que carga recebe a maior
potência? Para um máximo, minimiza-se $-P$.

📖 [capítulo 6 · Mesmo método, outra área](https://lacouth.github.io/metodos_telecom-site/unidade3-raizes/06-otimizacao/#mesmo-metodo-outra-area)

**✍️ Passo 5.** Escreva `menos_potencia(R)` e ache o mínimo dela: varredura em `np.linspace(1, 300, 300)` com `np.argmin`, e depois 8 passos de Newton para otimização.

In [ ]:
# ✍️ passo 5

**Preveja:** que resistência de carga tira a maior potência da fonte?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

$R = 50$ Ω — **igual** à resistência da fonte —, com 0,72 W. É o teorema da
máxima transferência de potência, e o motivo de cabos e antenas terem
impedâncias "casadas" (50 Ω, 75 Ω). O código é o mesmo da lata.

📖 [capítulo 6 · Mesmo método, outra área](https://lacouth.github.io/metodos_telecom-site/unidade3-raizes/06-otimizacao/#mesmo-metodo-outra-area)

</details>

## 🎯 Prática

Retoma o bloco *1. Primeiro, uma varredura*.
📖 [capítulo 6 · Primeiro, uma varredura](https://lacouth.github.io/metodos_telecom-site/unidade3-raizes/06-otimizacao/#primeiro-uma-varredura)

### 🎯 Sua vez — O máximo na grade

Escreva `maximo_na_grade(f, a, b, n)`, que devolve o `x` da grade `np.linspace(a, b, n)` em que `f` é **maior**.

In [ ]:
def maximo_na_grade(f, a, b, n):
    # sua solução aqui
    pass

In [ ]:
def parabola(x):
    return -(x - 2.3)**2


confere(maximo_na_grade, [
    ((parabola, 0, 5, 51), 2.3000000000000003),
])

<details>
<summary><b>💡 Dica</b></summary>

Como no passo 1, com `np.argmax` no lugar de `np.argmin`.

</details>

## 🧩 Resolvendo o problema

> *"**Qual o ângulo ideal para ela, e quantos centímetros ela ganha em relação aos
> 45°?**"* — o treinador.

A célula 📦 tem o alcance do arremesso, `alcance(theta, v, h)`, com o ângulo em
graus. A atleta solta o peso a 13,7 m/s, de 2,1 m de altura.

In [ ]:
# 📦 dados prontos — só rode esta célula
# Alcance (m) de um arremesso de peso com velocidade de saída v (m/s), saindo da
# altura h (m), com ângulo theta em GRAUS (sem arrasto: o peso é pesado e lento).
G = 9.81


def alcance(theta, v, h):
    th = theta * np.pi / 180
    return v * np.cos(th) / G * (v * np.sin(th) + np.sqrt(v**2 * np.sin(th)**2 + 2 * G * h))

### 🎯 Sua vez — O ângulo ótimo

Escreva `angulo_otimo(v, h)`, que acha o ângulo (em graus) de **maior** alcance:
varredura de 10° a 80° (71 pontos) com `np.argmin` em $-\text{alcance}$, e
depois 10 passos de Newton para otimização (`passo = 1e-4`).

In [ ]:
def angulo_otimo(v, h):
    # sua solução aqui
    pass

In [ ]:
confere(angulo_otimo, [
    ((13.7, 2.1), 42.161943449566984),
    ((10, 1.8), 40.68424638872446),
], tol=1e-5)

<details>
<summary><b>💡 Dica</b></summary>

Dentro da função, defina `menos_alcance(theta)` que devolve
`-alcance(theta, v, h)`. Depois é o passo 5, com ângulos no lugar de
resistências.

</details>

A resposta para o treinador:

In [ ]:
theta = angulo_otimo(13.7, 2.1)
if theta is not None:
    print("ângulo ótimo:", theta, "graus")
    print("alcance no ângulo ótimo:", alcance(theta, 13.7, 2.1), "m")
    print("alcance a 45 graus:     ", alcance(45, 13.7, 2.1), "m")

<details>
<summary><b>▶ O que os números dizem</b></summary>

O ângulo ótimo é **42.2°**, e não 45°: como o peso sai de 2,1 m de
altura, ele já "ganha" tempo de voo, e compensa jogar um pouco mais para a frente.
O alcance sobe de 21.04 m para 21.13 m:
**9 cm** a mais, sem nenhuma
força extra.

Na prática, os arremessadores de elite saem com ângulos ainda menores (entre 35° e
40°): o corpo humano consegue imprimir **mais velocidade** em ângulos baixos, e o
modelo, que fixa a velocidade, não sabe disso. É o de sempre: o método resolve o
modelo, e o modelo aproxima o mundo.

</details>

## 📋 A lista

Abra a [Lista 06](https://lacouth.github.io/metodos_telecom-site/listas/lista06/). O **Exercício 01** é à mão (✏️): uma parábola, uma
varredura e um passo de Newton. Comece por ele, no papel.

**a)** Por que um único passo de Newton acerta o mínimo de uma parábola **exatamente**?

<details>
<summary><b>▶ Resposta</b></summary>

Porque Newton para otimização aproxima $f$ por uma parábola e salta para o vértice dela. Se $f$ já é uma parábola, a aproximação é exata.

</details>

Termine o exercício e siga para o **Exercício 02**, a varredura como função.

## 🚪 Antes de sair

**1.** Por que otimizar é "achar uma raiz"? Raiz de quê?

<details>
<summary><b>▶ Resposta da 1</b></summary>

No mínimo e no máximo de uma função suave, a derivada é zero. Otimizar $f$ é achar uma raiz de $f'$.

</details>

**2.** Para que serve a varredura, se Newton é muito mais preciso?

<details>
<summary><b>▶ Resposta da 2</b></summary>

Para ver a paisagem inteira e dar um bom chute: Newton sozinho vai para o ponto de $f' = 0$ mais próximo, que pode ser um máximo ou um vale raso.

</details>

**3.** Como achar o **máximo** de uma função com um método feito para achar mínimos?

<details>
<summary><b>▶ Resposta da 3</b></summary>

Minimizando $-f$: o ponto mais alto de $f$ é o ponto mais baixo de $-f$.

</details>

## 🏠 Para casa

- Refaça no papel um passo de Newton para otimização numa parábola **sem olhar**.
- Termine a [Lista 06](https://lacouth.github.io/metodos_telecom-site/listas/lista06/).
- A Unidade 3 termina aqui: é hora de revisar os capítulos 1 a 6 para a Avaliação 1.